# 03 RQ3: smart money

Secondary test of RQ3 (`docs/hypotheses.md` Section 4). Track records, smart wallets
and the signal are fixed in D6 and built in `sql/03_marts/02_mart_wallet_signal.sql`;
the estimation details and Section 5 for RQ3 are proposed in a D9 draft that is not
yet in Section 9. The gate stays closed until D9 is committed.

- **Signal:** net USD that smart wallets bought in token1 minus token2 in the 7 days up
  to the 1-day snapshot trade, scaled to [−1, 1] by their total volume. It is 0 when
  there is no smart volume (D6.6).
- **Test:** in `token1_won ~ logit(p) + signal`, the signal coefficient is positive.
  1-day horizon, two-sided, α = 0.05, CR1 cluster-robust SEs. A positive coefficient
  means smart wallets know something the price does not.
- **Secondary:** maker-only and taker-only signals, and the test without bots, each
  with unadjusted p-values (D6.8, D6.9).

`RUN_CONFIRMATORY` controls whether the outcome column is loaded at all; with it
False, only the planted control on simulated outcomes runs.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.special import expit, logit
from statsmodels.tools.sm_exceptions import ConvergenceWarning, PerfectSeparationWarning

RUN_CONFIRMATORY = False   # False: outcome column never loaded (planted control only)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

SEED = 20261112          # D7.5, reused
CLIP = (0.001, 0.999)    # Section 4
MIN_CLUSTERS = 30        # Section 4
pd.set_option("display.width", 200)

SIGNALS = ["signal", "signal_maker", "signal_taker", "signal_nofloor", "signal_nobot",
           "signal_no_mm", "signal_no_tk", "signal_no_mx"]
cols = (["market_id", "horizon", "is_primary", "snap_price", "volume_usd", "smart_usd",
         "cluster_id", "event_id", "format", "subject", "close_year", "early_close"]
        + SIGNALS + (["token1_won"] if RUN_CONFIRMATORY else []))
frame = pd.read_parquet(DATA / "mart_analysis.parquet", columns=cols)
d = frame[frame["is_primary"] & (frame["horizon"] == "1d")].reset_index(drop=True)
vwap = pd.read_parquet(DATA / "mart_vwap.parquet")[["market_id", "horizon", "vwap_price"]]
d = d.merge(vwap, on=["market_id", "horizon"], how="left", validate="one_to_one")
if RUN_CONFIRMATORY:
    d["y"] = d["token1_won"].astype(float)
d["has_smart"] = d["smart_usd"] > 0
d.groupby("format").agg(markets=("has_smart", "size"), clusters=("cluster_id", "nunique"),
                        with_smart_volume=("has_smart", "sum"), mean_signal=("signal", "mean"))

,markets,clusters,with_smart_volume,mean_signal
format,,,,
event,197712,16644,134995,0.020063
template,153823,674,138365,0.153734


## 1. Estimation (D9 draft)

- **Model.** `token1_won ~ logit(p) + signal`, fitted by maximum likelihood (statsmodels
  `Logit`), with p clipped to [0.001, 0.999]. SEs are CR1 cluster-robust on the
  Section 3 clusters, with z reference distributions and 95% Wald CIs. A fit that
  fails to converge or shows separation is reported as not estimable (as D7.1).
- **Effect size.** The average change in win probability when the signal moves from 0
  to +1, holding each market's price: mean of expit(η + β) − expit(η), with η the
  fitted index at signal 0.
- **One confirmatory test**, so no multiplicity correction. The secondary tests carry
  unadjusted p-values; the refits and the sensitivities report CIs only.

In [2]:
def fit(df, sig_col="signal", price_col="snap_price", cluster_col="cluster_id", y=None):
    df = df[df[price_col].notna()]
    y = df["y"].to_numpy() if y is None else y
    x = logit(df[price_col].clip(*CLIP).to_numpy())
    s = df[sig_col].to_numpy()
    groups = pd.factorize(df[cluster_col])[0]
    out = {"n": len(df), "clusters": int(groups.max() + 1)}
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        res = sm.Logit(y, np.column_stack([np.ones_like(x), x, s])).fit(
            disp=0, maxiter=200, cov_type="cluster", cov_kwds={"groups": groups})
    bad = any(issubclass(w.category, (ConvergenceWarning, PerfectSeparationWarning)) for w in caught)
    b, se = res.params, res.bse
    eta0 = b[0] + b[1] * x
    return {**out, "estimable": bool(res.mle_retvals["converged"]) and not bad,
            "coef": b[2], "se": se[2], "ci_low": b[2] - 1.959964 * se[2],
            "ci_high": b[2] + 1.959964 * se[2], "p_value": res.pvalues[2],
            "slope_logit_p": b[1], "ame_0_to_1": np.mean(expit(eta0 + b[2]) - expit(eta0))}

## 2. Planted control (simulated outcomes only)

The real prices, signals and clusters of the 351,535 one-day markets are kept; only the
outcome is simulated, from y ~ Bernoulli(expit(logit(p) + β · signal)).

- **Planted:** β = 0.03, so the true signal coefficient is 0.03. This is a small
  effect, about four standard errors, chosen near the smallest the test can detect.
  A large β would pass trivially and tell us little.
- **Null:** β = 0.

Each scenario is drawn 200 times. Pass: the 95% CI covers the true β in at least 90%
of draws in both scenarios, β = 0.03 is rejected in at least 80% of draws, and the
null rejection rate is at most 0.08. The minimum detectable effect (80% power,
α = 0.05) is printed as 2.8 × the null SE.

In [3]:
rng = np.random.default_rng(SEED)
R = 200
eta = logit(d["snap_price"].clip(*CLIP).to_numpy())
rows = []
for name, beta in {"planted": 0.03, "null": 0.0}.items():
    p_true = expit(eta + beta * d["signal"].to_numpy())
    for r in range(R):
        res = fit(d, y=rng.binomial(1, p_true).astype(float))
        rows.append({"scenario": name, "rep": r, "truth": beta, **res})
pc = pd.DataFrame(rows)
pc["covers"] = (pc["ci_low"] <= pc["truth"]) & (pc["truth"] <= pc["ci_high"])
pc["rejects_0"] = pc["p_value"] < 0.05
summary = pc.groupby("scenario").agg(
    truth=("truth", "first"), mean_coef=("coef", "mean"), mean_se=("se", "mean"),
    sd_coef=("coef", "std"), coverage=("covers", "mean"), reject_rate=("rejects_0", "mean"),
    all_estimable=("estimable", "all"))
summary.to_csv(RESULTS / "rq3_planted_control.csv")
summary

,truth,mean_coef,mean_se,sd_coef,coverage,reject_rate,all_estimable
scenario,,,,,,,
null,0.00,-0.000029,0.007503,0.007299,0.955,0.045,True
planted,0.03,0.030586,0.007537,0.007651,0.930,0.990,True


In [4]:
checks = {
    "coverage >= 0.90 in both scenarios": bool((summary["coverage"] >= 0.90).all()),
    "planted beta rejected in >= 80% of draws": bool(summary.loc["planted", "reject_rate"] >= 0.80),
    "null rejection rate <= 0.08": bool(summary.loc["null", "reject_rate"] <= 0.08),
    "every fit estimable": bool(summary["all_estimable"].all()),
}
for name, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + name)
print(f"minimum detectable effect (log-odds per unit signal): {2.8 * summary.loc['null', 'mean_se']:.4f}")
PLANTED_CONTROL_PASSED = all(checks.values())

PASS coverage >= 0.90 in both scenarios
PASS planted beta rejected in >= 80% of draws
PASS null rejection rate <= 0.08
PASS every fit estimable
minimum detectable effect (log-odds per unit signal): 0.0210


## 3. Confirmatory and secondary tests (gated)

The confirmatory test uses the combined signal (D6). The secondary tests use the
maker-only signal, the taker-only signal (D6.8) and the signal without bots (D6.9),
each in its own model of the same form, with unadjusted p-values.

In [5]:
if RUN_CONFIRMATORY and PLANTED_CONTROL_PASSED:
    specs = {"confirmatory: all smart legs": "signal", "secondary: maker legs": "signal_maker",
             "secondary: taker legs": "signal_taker", "secondary: no bots": "signal_nobot"}
    tests = pd.DataFrame([{"test": k, "signal_col": v, **fit(d, v)} for k, v in specs.items()])
    tests.to_csv(RESULTS / "rq3_tests.csv", index=False)
    display(tests.round(5))
else:
    print("tests skipped (gate closed)")

tests skipped (gate closed)


## 4. Descriptives (gated)

- **By signal bin.** Markets with no smart volume, then signal in [−1, −0.5), [−0.5, 0),
  [0, 0.5) and [0.5, 1]. Per bin: markets, clusters, mean price, win rate, and mean
  (won − price) with a CR1 cluster-robust 95% CI (intercept-only OLS, as D8.1).
- **Refits** (coefficient and CI only):
  - no USD floor (D6.4);
  - markets with smart volume only (D6.6);
  - each bot group removed (D6.9);
  - each format separately.

In [6]:
def mean_resid(df):
    r = sm.OLS((df["y"] - df["snap_price"]).to_numpy(), np.ones(len(df))).fit(
        cov_type="cluster", cov_kwds={"groups": pd.factorize(df["cluster_id"])[0]})
    b, se = r.params[0], r.bse[0]
    return {"markets": len(df), "clusters": df["cluster_id"].nunique(),
            "mean_price": df["snap_price"].mean(), "win_rate": df["y"].mean(),
            "resid": b, "ci_low": b - 1.959964 * se, "ci_high": b + 1.959964 * se}


if RUN_CONFIRMATORY:
    s = d["signal"]
    d["signal_bin"] = np.select(
        [~d["has_smart"], s < -0.5, s < 0, s < 0.5],
        ["no smart volume", "[-1, -0.5)", "[-0.5, 0)", "[0, 0.5)"], "[0.5, 1]")
    order = ["no smart volume", "[-1, -0.5)", "[-0.5, 0)", "[0, 0.5)", "[0.5, 1]"]
    bins = pd.DataFrame([{"signal_bin": b, **mean_resid(d[d["signal_bin"] == b])} for b in order])
    bins.to_csv(RESULTS / "rq3_signal_bins.csv", index=False)

    refit_specs = {
        "D6.4 no USD floor": (d, "signal_nofloor"),
        "D6.6 smart volume only": (d[d["has_smart"]], "signal"),
        "D6.9 no market-maker-like bots": (d, "signal_no_mm"),
        "D6.9 no taker-like bots": (d, "signal_no_tk"),
        "D6.9 no mixed bots": (d, "signal_no_mx"),
        "format: event": (d[d["format"] == "event"], "signal"),
        "format: template": (d[d["format"] == "template"], "signal"),
    }
    keep = ["n", "clusters", "estimable", "coef", "ci_low", "ci_high", "ame_0_to_1"]
    refits = pd.DataFrame([{"refit": k, **{c: v for c, v in fit(df, sc).items() if c in keep}}
                           for k, (df, sc) in refit_specs.items()])
    refits.to_csv(RESULTS / "rq3_refits.csv", index=False)
    display(bins.round(5), refits.round(5))
else:
    print("descriptives skipped (gate closed)")

descriptives skipped (gate closed)


## 5. Sensitivities (Section 5, D9.4 draft; gated)

The confirmatory model is refitted under each variant; each reports its coefficient
and 95% CI only.

- **5.2** keeps markets with at least $10,000 volume.
- **5.3** uses the 5-minute VWAP as p. Markets with no VWAP drop out, and the signal
  is unchanged.
- **5.5** clusters by `event_id`.
- **5.6** fits each `closedTime` year with at least 30 clusters.
- **5.7** drops early closes (a fragility check; see D8.6).

5.1 and 5.4 are not applied, because the signal exists only for primary 1-day rows
(D9.4).

In [7]:
if RUN_CONFIRMATORY:
    sens_specs = {
        "5.2 volume >= $10k": (d[d["volume_usd"] >= 10_000], "snap_price", "cluster_id"),
        "5.3 5-min VWAP": (d, "vwap_price", "cluster_id"),
        "5.5 event clustering": (d, "snap_price", "event_id"),
        "5.7 no early closes": (d[~d["early_close"].astype(bool)], "snap_price", "cluster_id"),
    }
    for year in sorted(d["close_year"].unique()):
        sens_specs[f"5.6 year {year}"] = (d[d["close_year"] == year], "snap_price", "cluster_id")
    rows = []
    for name, (df, pcol, ccol) in sens_specs.items():
        G = df.loc[df[pcol].notna(), ccol].nunique()
        if G < MIN_CLUSTERS:
            rows.append({"sensitivity": name, "clusters": G, "fitted": False})
            continue
        r = fit(df, "signal", pcol, ccol)
        rows.append({"sensitivity": name, "fitted": r["estimable"], "n": r["n"], "clusters": r["clusters"],
                     "coef": r["coef"] if r["estimable"] else np.nan,
                     "ci_low": r["ci_low"] if r["estimable"] else np.nan,
                     "ci_high": r["ci_high"] if r["estimable"] else np.nan})
    sens = pd.DataFrame(rows)
    sens.to_csv(RESULTS / "rq3_sensitivities.csv", index=False)
    display(sens.round(5))
else:
    print("sensitivities skipped (gate closed)")

sensitivities skipped (gate closed)
